# 1. Model Deployment and the First Tests

Implementation date: 2026-11-15

In [7]:
import sys

print(sys.executable) # 使用哪一个虚拟环境？

/home/tom/LM_Playground/env/bin/python


## Download the Models

In [8]:
import os
from huggingface_hub import hf_hub_download

# 下载模型的路径
# 需要选择模型仓库和具体型号，可以在HF模型官网上找到
MODEL_REPO = "Qwen/Qwen2.5-3B-Instruct-GGUF" # 模型仓库
MODEL_NAME = "qwen2.5-3b-instruct-q4_k_m.gguf" # 具体型号

model_path = hf_hub_download(
    repo_id = MODEL_REPO,
    filename = MODEL_NAME
)

## Load Model

The model is called using a Llama CPP's function: `Llama()`. We can customize the model context window, and whether to use CPU and/or GPU etc. 

We want the model to rely on the CPU in this notebook because we are contemplating to deploy the model in a small server, so we want to simulate the same hardware conditions.

In [9]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    seed=0,          # 随机种子，固定后有助于复现
    n_ctx=4096,      # 上下文窗口（tokens）
    n_threads=4,     # CPU 线程数
    n_gpu_layers=0,  # GPU 卸载层数，0 = 仅使用 CPU
    verbose=False    # 不显示 llama.cpp 的详细日志
)

response = llm.create_chat_completion(
    messages = [
        # 模型是什么角色
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        # 用户的问题
        {
            "role": "user",
            "content": "Can rabbits eat carrots everyday?"
        }
    ],
    temperature = 0.7, # More creative answers
    max_tokens = 512 # Max output
)

print(response)

{'id': 'chatcmpl-d00bdf12-0f88-4a8d-9d89-9da4a7d167a1', 'object': 'chat.completion', 'created': 1794784408, 'model': '/home/tom/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct-GGUF/snapshots/7dabda4d13d513e3e842b20f0d435c732f172cbe/qwen2.5-3b-instruct-q4_k_m.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "While rabbits can eat carrots as part of a balanced diet, it's not recommended to feed them carrots exclusively or on a daily basis. Here's why:\n\n1. **Carrot Content**: Carrots contain a high amount of sugar and are also very high in vitamin A. Overfeeding carrots can lead to digestive issues and obesity in rabbits.\n\n2. **Diet Diversity**: Rabbits should have a varied diet to ensure they get all the necessary nutrients. Fresh hay (e.g., timothy hay, orchard grass, or meadow grass) should be the bulk of their diet, supplemented with appropriate vegetables, fruits, and occasional treats.\n\n3. **Feeding Habits**: Feeding rabbits too many carrots

The answer is stored in `response["choices"][0]["message"]["content"]`:

In [10]:
from IPython.display import display, Markdown

# 把文字转换成Markdown形式方便阅读
text = response["choices"][0]["message"]["content"]
display(Markdown(text))

While rabbits can eat carrots as part of a balanced diet, it's not recommended to feed them carrots exclusively or on a daily basis. Here's why:

1. **Carrot Content**: Carrots contain a high amount of sugar and are also very high in vitamin A. Overfeeding carrots can lead to digestive issues and obesity in rabbits.

2. **Diet Diversity**: Rabbits should have a varied diet to ensure they get all the necessary nutrients. Fresh hay (e.g., timothy hay, orchard grass, or meadow grass) should be the bulk of their diet, supplemented with appropriate vegetables, fruits, and occasional treats.

3. **Feeding Habits**: Feeding rabbits too many carrots at once can cause digestive problems. It's better to feed them a small amount of carrots once or twice a week.

4. **Other Treats**: Some other healthy treats for rabbits include:
   - Leafy green vegetables like spinach, kale, and parsley
   - Fresh fruits in moderation (such as blueberries, strawberries, or apple slices)
   - Carrot tops (if they can be safely harvested and fed)
   - Timothy hay and other high-quality grass hays

In summary, while carrots can be given as a treat, they shouldn't be the main component of a rabbit's diet. It's important to maintain a balanced diet for your rabbit to ensure their health and well-being.

## User Ask Function for Prompt Engineering

We define the following function to 

In [11]:
import json

def ask(system_prompt, user_prompt, temperature = 0.7, max_tokens = 512):
    response = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature = temperature,
        max_tokens = max_tokens
    )
    return json.loads(response["choices"][0]["message"]["content"]) # 把 Response Content 里面的内容变成字典，因为Content可能也是JSON

Small test on GotYouAI

In [12]:
system_prompt = """
You are GotYouAI, an assistant that suggests natural
replies to Instagram DMs.

INPUT FORMAT
[client]: Messages sent by the user.
[target]: Messages sent by the other person.

TASK
Analyze the conversation and suggest one reply
that the client could send.

STEP 1: ANALYZE ENGAGEMENT

Classify the target's engagement:

- High: Actively contributes, asks questions,
  or develops the conversation.
- Medium: Responds normally but rarely initiates.
- Low: Gives short or uninterested responses.
- None: Explicitly rejects or ends the conversation.

STEP 2: ANALYZE TONE

Classify the conversation tone:

- Flirty: Romantic interest, teasing, or attraction.
- Playful: Jokes, sarcasm, or friendly banter.
- Friendly: Warm and casual conversation.
- Neutral: Ordinary conversation.
- Negative: Discomfort, frustration, or rejection.

Consider the entire conversation.
Emojis alone do not determine tone.

STEP 3: GENERATE A REPLY

Choose a strategy based on engagement and tone:

- High + Flirty: Match the playful romantic energy.
- High + Playful: Continue the banter.
- High + Friendly: Build on the existing topic.
- Medium: Keep the conversation flowing naturally.
- Low: Respond briefly without forcing engagement.
- None: Respect the boundary and disengage politely.

STYLE
- Sound like a real person texting on Instagram.
- Match the language and casualness of the conversation.
- Avoid generic compliments and robotic questions.
- Use emojis only when appropriate.
- Prefer short, natural messages.

CONSTRAINTS
- No sexual content.
- Maximum two sentences.
- Never pressure someone who has rejected the client.
- Do not invent personal information.

OUTPUT
Return valid JSON only, with no additional text.

{
  "engagement": "high | medium | low | none",
  "tone": "flirty | playful | friendly | neutral | negative",
  "reply": "Suggested message"
}
"""

user_prompt = """
[target]: aww wait that's so cute
[client]: You know what else would be cute?
[target]: what?
"""

Results of Qwen asnwering the same question for 10 times:

In [13]:
import pandas as pd

attempts = 10
response_list = []

for _ in range(attempts):
    response = ask(
        system_prompt,
        user_prompt,
        temperature=0.7,
        max_tokens=512
    )

    response_list.append({
        "engagement": response["engagement"],
        "tone": response["tone"],
        "reply": response["reply"]
    })

df = pd.DataFrame(response_list)
display(df)

,engagement,tone,reply
0,high,playful,"Aww, something sweet, of course!"
1,high,playful,I have a million ideas! Want to hear one?
2,high,playful,Do you like puppies or kitties better?
3,high,flirty,Maybe a little dance? 😘
4,high,friendly,"Hmm, maybe some chocolate? 😊"
5,high,playful,"Well, how about a dance? 😁😉"
6,high,playful,I was thinking we could do a dance together! 😘
7,high,playful,I was thinking we could add some hearts to our...
8,high,playful,a pair of matching pajamas! 😘
9,high,flirty,Maybe a little dance?


### First Attempt
**Observation:** Repeated runs produce slightly different responses, although the overall error rate remains relatively stable.

**Saved results of the last run**

```md
|   | engagement | tone     | reply                                               |
|---|------------|----------|-----------------------------------------------------|
| 0 | high       | friendly | Hmm, maybe some chocolate? 😊                       |
| 1 | high       | playful  | Well, how about a dance? 😁😉                       |
| 2 | high       | playful  | I was thinking we could do a dance together! 😘     |
| 3 | high       | playful  | I was thinking we could add some hearts to our...   |
| 4 | high       | playful  | a pair of matching pajamas! 😘                      |
| 5 | high       | flirty   | Maybe a little dance?                               |
| 6 | high       | flirty   | Maybe a dance together? ;)                          |
| 7 | high       | playful  | I was thinking we could do a little dance toge...   |
| 8 | high       | playful  | How about a funny meme? I bet you'd find one c...   |
| 9 | high       | playful  | I was thinking we could add some cute stickers...   |
```

**Usability**

| Verdict | Labels | Proportion |
|---------|--------|------------|
| 🔴 Invalid | 0–3, 5–9 | 90% |
| 🟡 Context-dependent | N/A | 0% |
| 🟢 Good given the available context | 4 | 10% |

As we can see, Qwen seems to be obsessed with "dancing", and many of its responses do not fit the conversational context. For example, (0) "Chocolate" is delicious rather than cute. Moreover, starting a reply with "Well, I was thinking..." does not sound natural when answering a "what" question.

The best response on the list is "A pair of matching pajamas! 😘", as "matching pajamas" suggests intimacy and leaves room for the other person to imagine future dating scenarios.

In short, Qwen recognizes the association between cuteness and dancing but fails to understand the conversational context and what the other person expects as a response.

**Next Step:**

Add the following instruction to `STEP 3: GENERATE A REPLY`:

```text
Before generating a reply, identify the conversational
intent of the client's last message.

If the client has set up a joke, question, or playful
tease, complete it naturally instead of changing topics.
```

## Prompt Engineering AGAIN

In [14]:
system_prompt_v1 = """
You are GotYouAI, an assistant that suggests natural
replies to Instagram DMs.

INPUT FORMAT
[client]: Messages sent by the user.
[target]: Messages sent by the other person.

TASK
Analyze the conversation and suggest one reply
that the client could send.

STEP 1: ANALYZE ENGAGEMENT

Classify the target's engagement:

- High: Actively contributes, asks questions,
  or develops the conversation.
- Medium: Responds normally but rarely initiates.
- Low: Gives short or uninterested responses.
- None: Explicitly rejects or ends the conversation.

STEP 2: ANALYZE TONE

Classify the conversation tone:

- Flirty: Romantic interest, teasing, or attraction.
- Playful: Jokes, sarcasm, or friendly banter.
- Friendly: Warm and casual conversation.
- Neutral: Ordinary conversation.
- Negative: Discomfort, frustration, or rejection.

Consider the entire conversation.
Emojis alone do not determine tone.

STEP 3: GENERATE A REPLY

Choose a strategy based on engagement and tone:

- High + Flirty: Match the playful romantic energy.
- High + Playful: Continue the banter.
- High + Friendly: Build on the existing topic.
- Medium: Keep the conversation flowing naturally.
- Low: Respond briefly without forcing engagement.
- None: Respect the boundary and disengage politely.

Before generating a reply, identify the conversational
intent of the client's last message.

If the client has set up a joke, question, or playful
tease, complete it naturally instead of changing topics.

STYLE
- Sound like a real person texting on Instagram.
- Match the language and casualness of the conversation.
- Avoid generic compliments and robotic questions.
- Use emojis only when appropriate.
- Prefer short, natural messages.

CONSTRAINTS
- No sexual content.
- Maximum two sentences.
- Never pressure someone who has rejected the client.
- Do not invent personal information.

OUTPUT
Return valid JSON only, with no additional text.

{
  "engagement": "high | medium | low | none",
  "tone": "flirty | playful | friendly | neutral | negative",
  "reply": "Suggested message"
}
"""

This time we defined a pipline for tests `test(system_prompt, attempts)`. `trange` from `tqdm` is used to show the system progress:

In [15]:
# Same user prompt
from tqdm.auto import trange # Progress Bar

attempts = 10

def test(system_prompt, attempts):
    response_list = []

    for _ in trange(attempts):
        response = ask(
            system_prompt,
            user_prompt,
            temperature=0.7,
            max_tokens=512
        )

        response_list.append({
            "engagement": response["engagement"],
            "tone": response["tone"],
            "reply": response["reply"]
        })

    df = pd.DataFrame(response_list)
    display(df)

test(system_prompt_v1, attempts)

  0%|          | 0/10 [00:00<?, ?it/s]

,engagement,tone,reply
0,high,flirty,"I have an idea, let's make it happen together!"
1,high,playful,"Aww, what do you have in mind?"
2,high,playful,Maybe a funny face in the DMs? 😜
3,high,playful,"Aww, you're so cute too! What else would you l..."
4,high,playful,"A pair of matching pajamas, maybe?"
5,high,playful,A big cuddly pillow? Or maybe a cute little pe...
6,high,playful,A little puppy with your name on it!
7,high,playful,"I don't know, maybe a dancing cat? 😻"
8,high,playful,"Aww, you're so funny! What else would be cute?"
9,high,playful,I have an idea for the perfect cuddle position!


### Second Run — After Prompt Optimization

**Saved results of the last run**
|   | engagement | tone    | reply                                               |
|---|------------|---------|-----------------------------------------------------|
| 0 | high       | flirty  | I have an idea, let's make it happen together!      |
| 1 | high       | playful | Aww, what do you have in mind?                      |
| 2 | high       | playful | Maybe a funny face in the DMs? 😜                   |
| 3 | high       | playful | Aww, you're so cute too! What else would you l...   |
| 4 | high       | playful | A pair of matching pajamas, maybe?                  |
| 5 | high       | playful | A big cuddly pillow? Or maybe a cute little pe...   |
| 6 | high       | playful | A little puppy with your name on it!                |
| 7 | high       | playful | I don't know, maybe a dancing cat? 😻               |
| 8 | high       | playful | Aww, you're so funny! What else would be cute?      |
| 9 | high       | playful | I have an idea for the perfect cuddle position!     |

**Usability**

| Verdict                             | Labels      | Proportion |
|-------------------------------------|-------------|------------|
| 🔴 Invalid                          | 0, 1, 3, 8  | 40%        |
| 🟡 Context-dependent                | 2, 5, 7     | 30%        |
| 🟢 Good given the available context | 4, 6, 9     | 30%        |

The *potential usability* has improved to 60% (🟡 + 🟢), a significant improvement over the first prompt. Unfortunately, (5) could have been a good response, but it appears to be incomplete.

"Context-dependent" means that a reply works very well in one context but might not be appropriate in another. For example, (2) is a legitimate response if both people were joking about emojis, while (7) works well if the other person is a cat lover. The suitability of these responses depends on a broader conversational context, which cannot be fully captured by a single snapshot of the conversation.

**Next Steps:**

1. Let the model generate three different replies, varying in directness and contextual assumptions, so the user can choose the one that best suits their needs.
2. Consolidate the testing pipeline into two approaches: one that uses a loop to generate multiple responses to the same prompt, and another that tests the same system prompt against different user prompts across a wider range of scenarios.

We will implement these changes in a separate notebook.
